# 1.9 — Method Signatures 🧋

### AP CSA · Unit 1: Using Objects and Methods
**Boba Cafe Series · Lesson 9**

---

> **Setup note:** every code cell runs on the **IJava kernel** (Java 17+). Check that the kernel picker says *Java*. Each cell declares a class and then calls it with `ClassName.main(null);`.

## Taking the first line apart

Across the last two lessons you've read lines like this one without ever being asked to explain every piece of it:

```java
public static double calculateTotal(int quantity, double unitPrice)
```

You could already *use* `calculateTotal` correctly. Today you learn to *read* that line fluently — what each word contributes, what part of it Java actually calls the "signature," and how Java decides which method you meant when two of them share a name.

That last question matters more than it sounds like it should. The manager asks you to add a new pricing method to the register, and there's already one called `drinkCost`. Can you have two? Which one runs when a customer's order comes in?

### What you'll be able to do by the end

| # | Objective | CED reference |
|---|---|---|
| 1 | Identify every part of a method header | 1.9.A |
| 2 | State precisely what a method's **signature** is — and is not | 1.9.A |
| 3 | Call a method by matching arguments to parameters in type and order | 1.9.A |
| 4 | Distinguish `void` methods from methods that return a value | 1.9.A |
| 5 | Explain what happens when a returned value is stored, used, or ignored | 1.9.A |
| 6 | Explain **method overloading** and how Java picks the right overload | 1.9.A |

---

## Part 1 — Anatomy of a method header

Every method you call at the cafe — `Math.sqrt`, `scan.nextInt`, or one you write yourself — starts with a **header** built from the same pieces, in the same order.

```
  public   static   double   calculateTotal   (int quantity, double unitPrice)
  ^^^^^^   ^^^^^^   ^^^^^^   ^^^^^^^^^^^^^^^   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    |         |        |            |                        |
    |         |        |            |                        └─ PARAMETER LIST:
    |         |        |            |                           what the caller must supply
    |         |        |            └─ NAME: what you call it
    |         |        └─ RETURN TYPE: the type of value handed back
    |         └─ static: called on the class itself, no object needed (Unit 3 explains the alternative)
    └─ ACCESS MODIFIER: who is allowed to call this method
```

| Piece | This example | What it tells you |
|---|---|---|
| Access modifier | `public` | Any code anywhere can call this method |
| `static` | `static` | Call it as `ClassName.method(...)`, not through an object |
| Return type | `double` | Calling this method produces a `double` value |
| Name | `calculateTotal` | What you type to call it |
| Parameter list | `(int quantity, double unitPrice)` | Exactly what you must hand it, in this order |

Every one of those five pieces is doing a job. Miss one and either the compiler stops you, or the method doesn't do what you expected.

In [ ]:
public class HeaderTour {

    /**
     * Calculates the total cost of an order, including 7.75% sales tax.
     * @param quantity the number of drinks ordered
     * @param unitPrice the price of one drink, in dollars
     * @return the total cost in dollars, tax included
     */
    public static double calculateTotal(int quantity, double unitPrice) {
        double subtotal = quantity * unitPrice;
        return subtotal + subtotal * 0.0775;
    }

    public static void main(String[] args) {
        double total = calculateTotal(3, 5.75);
        System.out.println("3 drinks at $5.75, with tax: $" + total);
    }
}

HeaderTour.main(null);

---

## Part 2 — What "signature" precisely means

Here's the sentence to memorize, because the exact wording matters on the exam:

> **A method's signature is its name plus the type and order of its parameters. Nothing else.**

```
    calculateTotal(int, double)
    ^^^^^^^^^^^^^^  ^^^  ^^^^^^
       name          parameter TYPES, in order
```

| Part of the header | In the signature? |
|---|---|
| Access modifier (`public`) | **No** |
| `static` | **No** |
| Return type (`double`) | **No** |
| Method name | **Yes** |
| Parameter *types*, in order | **Yes** |
| Parameter *names* | **No** |

That last row is the one that surprises people. `calculateTotal(int quantity, double unitPrice)` and `calculateTotal(int drinkCount, double pricePerDrink)` have the **exact same signature** — `calculateTotal(int, double)`. The parameter names exist purely to make the method's own code readable. A caller never types them and Java never looks at them when matching a call.

Why the return type isn't part of the signature is Part 6's problem — it's the source of a genuine trap.

In [ ]:
public class SameSignature {

    // These two headers use different parameter NAMES,
    // but they have the IDENTICAL signature: tallyOrder(int, double)

    public static double tallyOrder(int quantity, double unitPrice) {
        return quantity * unitPrice;
    }

    public static void main(String[] args) {
        // The caller never mentions "quantity" or "unitPrice" -- only VALUES, in order.
        System.out.println("Order total: $" + tallyOrder(4, 3.25));
    }
}

SameSignature.main(null);

**Heads up: the next cell is supposed to fail.** It tries to declare two methods that share a signature — only the return type differs. Read the compiler's message carefully.

In [ ]:
public class DuplicateSignature {

    public static int stockCount(int cupsPerCase) {
        return cupsPerCase * 24;
    }

    // Same name, same parameter TYPE and order: stockCount(int).
    // A different return type does NOT make this a different signature.
    public static double stockCount(int cupsPerCase) {
        return cupsPerCase * 24.0;
    }

    public static void main(String[] args) {
        System.out.println(stockCount(2));
    }
}

DuplicateSignature.main(null);

`method stockCount(int) is already defined` — Java doesn't even mention the return type in that message, because **the return type was never part of what made these two methods "different" in the first place.** This is the trap: it looks like overloading, and it isn't.

---

## Part 3 — Calling a method: matching by type and order

When you call a method, Java lines up your **arguments** — the actual values you supply — against the **parameters** in the header, strictly by position.

```
   calculateTotal(3, 5.75)
   header:  (int quantity, double unitPrice)
                  |              |
              3 -> quantity   5.75 -> unitPrice
```

Swap the order and the types no longer line up.

In [ ]:
public class ArgumentOrder {

    public static String describeOrder(int quantity, String drink) {
        return quantity + "x " + drink;
    }

    public static void main(String[] args) {
        System.out.println(describeOrder(3, "Taro Milk Tea"));   // correct order
    }
}

ArgumentOrder.main(null);

**Heads up: the next cell is supposed to fail.** The arguments are swapped.

In [ ]:
public class WrongOrder {

    public static String describeOrder(int quantity, String drink) {
        return quantity + "x " + drink;
    }

    public static void main(String[] args) {
        System.out.println(describeOrder("Taro Milk Tea", 3));   // swapped!
    }
}

WrongOrder.main(null);

The compiler doesn't guess what you meant. `describeOrder(int, String)` was declared; `describeOrder(String, int)` was called. Those are different signatures, and no method matches.

**Argument count also has to match exactly** — no optional parameters exist in Java the way they do in some other languages. Every parameter in the header needs exactly one argument in the call.

---

## Part 4 — `void` versus a value

The return type `void` is a special case: it means **this method performs an action and hands nothing back.**

| | `void` method | Value-returning method |
|---|---|---|
| Return type | `void` | An actual type: `int`, `double`, `String`, ... |
| Ends with | No `return` needed (or a bare `return;` to exit early) | A `return value;` statement, required on every path |
| Called as | Its own statement | Part of an expression, or its own statement |
| Can you write `int x = method();`? | **No** | Yes, if the type matches |

`System.out.println(...)` is `void` — it prints, then hands nothing back. `Math.sqrt(...)` returns a `double` — it hands a value back for you to use.

In [ ]:
public class VoidVsReturn {

    // VOID: performs an action (printing), returns nothing
    public static void printReceipt(String drink, double price) {
        System.out.println(drink + ": $" + price);
    }

    // RETURN TYPE double: produces a value, does not print anything itself
    public static double applyDiscount(double price, double percentOff) {
        return price - price * percentOff;
    }

    public static void main(String[] args) {
        printReceipt("Matcha Latte", 6.50);              // called as its own statement

        double discounted = applyDiscount(6.50, 0.10);   // the VALUE is stored
        System.out.println("After 10% off: $" + discounted);

        System.out.println("Or used directly: $" + applyDiscount(6.50, 0.20));
    }
}

VoidVsReturn.main(null);

**Heads up: the next cell is supposed to fail.** It tries to store the result of a `void` method.

In [ ]:
public class StoreTheVoid {

    public static void printReceipt(String drink, double price) {
        System.out.println(drink + ": $" + price);
    }

    public static void main(String[] args) {
        double result = printReceipt("Matcha Latte", 6.50);   // printReceipt returns NOTHING
    }
}

StoreTheVoid.main(null);

`incompatible types: void cannot be converted to double`. There is no value on the right side of that `=` at all — `void` isn't a placeholder for "nothing in particular," it's a guarantee that nothing comes back.

### The quieter mistake: ignoring a value that matters

The opposite mistake compiles perfectly and is much easier to miss.

In [ ]:
public class WastedReturn {

    public static double applyDiscount(double price, double percentOff) {
        return price - price * percentOff;
    }

    public static void main(String[] args) {
        System.out.println("Before: $6.50");

        applyDiscount(6.50, 0.10);      // computes the discounted price... and throws it away

        System.out.println("Still $6.50 -- the discount was never used!");
    }
}

WastedReturn.main(null);

That compiled with zero warnings and ran perfectly. `applyDiscount` did its job, computed `5.85`, and handed it back — and the calling code just didn't catch it. This is a **logic error**, the same silent category from every lesson so far: nothing crashed, nothing printed a warning, the discount is simply gone.

**A returned value that matters must be stored in a variable or used directly** — printed, passed to another method, or included in a larger expression.

---

## Part 5 — Overloading

> **Definition:** **Overloading** means writing multiple methods with the **same name** but **different signatures** — different parameter types, different parameter counts, or both.

You've already used overloaded methods without being told. Back in Lesson 1.7:

```java
Math.abs(-7);      // calls  static int abs(int x)
Math.abs(-7.5);    // calls  static double abs(double x)
```

Same name, `abs`. Different signatures: `abs(int)` and `abs(double)`. **Java decides which one you meant by looking at the types of the arguments you passed** — this happens at compile time, before the program ever runs.

Overloading exists so a family of closely related operations can share one intuitive name, instead of forcing you to remember `absInt`, `absDouble`, `absFloat`, and so on.

In [ ]:
public class OverloadedPricing {

    // drinkCost(int, double) -- flat price times quantity
    public static double drinkCost(int quantity, double unitPrice) {
        return quantity * unitPrice;
    }

    // drinkCost(int, double, boolean) -- same idea, with an optional topping fee
    public static double drinkCost(int quantity, double unitPrice, boolean extraTopping) {
        double base = quantity * unitPrice;
        return extraTopping ? base + 0.75 * quantity : base;
    }

    // drinkCost(String) -- a totally different lookup, same name
    public static double drinkCost(String drinkName) {
        if (drinkName.equals("Taro Milk Tea"))   return 5.75;
        if (drinkName.equals("Matcha Latte"))    return 6.50;
        return 5.00;    // default price
    }

    public static void main(String[] args) {
        System.out.println("2 @ $5.75:              $" + drinkCost(2, 5.75));
        System.out.println("2 @ $5.75, with pearls:  $" + drinkCost(2, 5.75, true));
        System.out.println("Lookup by name:          $" + drinkCost("Matcha Latte"));
    }
}

OverloadedPricing.main(null);

Three completely different signatures, one shared name: `drinkCost(int, double)`, `drinkCost(int, double, boolean)`, and `drinkCost(String)`. At each call site, Java looked at what was actually passed and picked the one and only method whose signature fit.

### Widening during overload matching

What happens if you pass an `int` where a method expects a `double`? From Lesson 1.4 you know `int` widens into `double` automatically — and overload matching honors that.

In [ ]:
public class WideningMatch {

    public static double halfOf(double x) {
        return x / 2;
    }

    public static void main(String[] args) {
        // No halfOf(int) exists. Java widens the int argument to a double
        // and matches halfOf(double) instead.
        System.out.println(halfOf(9));       // 9 widens to 9.0
        System.out.println(halfOf(9.0));     // already a double
    }
}

WideningMatch.main(null);

Both calls landed on the same method, `halfOf(double)`. Java will always widen a value to find a match before giving up — but it will never **narrow** one for you (turning a `double` into an `int` implicitly), for exactly the reason from Lesson 1.5.

---

# Practice: Reading the Register's Source Code

Four tasks, in order.

---

## Hack 1 — Label the header

For each piece of this method header, write what it is. Double-click to edit.

```java
public static int cupsNeeded(int customers, int drinksPerCustomer)
```

| Piece | What is it? |
|---|---|
| `public` | |
| `static` | |
| `int` (the first one) | |
| `cupsNeeded` | |
| `(int customers, int drinksPerCustomer)` | |

Then write the method's **signature** exactly as Java defines it — nothing more, nothing less.

**Signature:**

<details>
<summary><b>Check your answers</b></summary>

| Piece | What is it? |
|---|---|
| `public` | Access modifier |
| `static` | Called on the class itself, no object needed |
| `int` (first one) | Return type |
| `cupsNeeded` | Method name |
| `(int customers, int drinksPerCustomer)` | Parameter list |

**Signature:** `cupsNeeded(int, int)` — name plus parameter *types*, in order. Not the return type, not the parameter names.
</details>

---

## Hack 2 — Which overload runs?

Given these three overloaded methods, predict which one each call uses, and what it prints. Fill in your predictions **before** running the cell.

```java
public static String sizeLabel(int ounces)                 { return ounces + " oz"; }
public static String sizeLabel(int ounces, boolean isHot)   { return ounces + " oz, " + (isHot ? "hot" : "cold"); }
public static String sizeLabel(double ounces)                { return ounces + " oz (metric-ish)"; }
```

| Call | Which signature runs? | Your prediction |
|---|---|---|
| `sizeLabel(16)` | | |
| `sizeLabel(16, true)` | | |
| `sizeLabel(16.0)` | | |
| `sizeLabel(16.5)` | | |

In [ ]:
public class WhichOverload {

    public static String sizeLabel(int ounces) {
        return ounces + " oz";
    }

    public static String sizeLabel(int ounces, boolean isHot) {
        return ounces + " oz, " + (isHot ? "hot" : "cold");
    }

    public static String sizeLabel(double ounces) {
        return ounces + " oz (metric-ish)";
    }

    public static void main(String[] args) {
        System.out.println("1. " + sizeLabel(16));
        System.out.println("2. " + sizeLabel(16, true));
        System.out.println("3. " + sizeLabel(16.0));
        System.out.println("4. " + sizeLabel(16.5));
    }
}

WhichOverload.main(null);

<details>
<summary><b>Explanations</b></summary>

1. `sizeLabel(int)` — one plain `int` argument matches the one-parameter `int` version exactly.
2. `sizeLabel(int, boolean)` — two arguments, so only the two-parameter overload's signature fits.
3. `sizeLabel(double)` — the literal `16.0` **is** a `double`, so it matches `sizeLabel(double)` directly. It does **not** call `sizeLabel(int)`, because Java never narrows a `double` down to an `int` for you.
4. `sizeLabel(double)` — same reasoning; `16.5` can only be a `double`.

Row 3 is the one people get wrong. `16.0` looks like it "could" be 16, but its type is `double` the instant it's written with a decimal point, and overload matching cares about type, not mathematical value.
</details>

---

## Hack 3 — Fix the broken register

The cell below has **three** bugs related to this lesson. Run it, fix all three, then fill in the report.

In [ ]:
public class BrokenCalls {

    public static double subtotal(int quantity, double unitPrice) {
        return quantity * unitPrice;
    }

    public static void printLine(String label, double amount) {
        System.out.println(label + ": $" + amount);
    }

    public static void main(String[] args) {
        // Bug 1: arguments are in the wrong order for subtotal's signature
        double sub = subtotal(5.75, 3);

        // Bug 2: printLine is void -- nothing comes back to store
        double result = printLine("Subtotal", sub);

        // Bug 3: this call computes a discount but the result is never used
        applyDiscount(sub, 0.10);

        System.out.println("Final subtotal: $" + sub);
    }

    public static double applyDiscount(double amount, double percentOff) {
        return amount - amount * percentOff;
    }
}

BrokenCalls.main(null);

**Your bug report** (double-click to edit):

| Bug | What was wrong | Your fix |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |

<details>
<summary><b>Check your answers</b></summary>

**Bug 1 — argument order.** `subtotal(int quantity, double unitPrice)` expects the `int` first. The call `subtotal(5.75, 3)` puts a `double` where an `int` is expected — Java won't narrow it, so this doesn't compile. Fix:

```java
double sub = subtotal(3, 5.75);
```

**Bug 2 — storing a `void` return.** `printLine` returns nothing. Fix by calling it as its own statement:

```java
printLine("Subtotal", sub);
```

**Bug 3 — a wasted return value.** `applyDiscount` computes a real discounted price and it's thrown away; `sub` is never updated. Fix by storing or using the result:

```java
sub = applyDiscount(sub, 0.10);
```

Bug 1 is a compile-time error. Bugs 2 and 3 are opposite mistakes — one tries to grab a value that doesn't exist, the other lets a real value slip away — and both are about the same underlying idea: **what a method's return type actually promises you.**
</details>

---

## Hack 4 — Write an overloaded set

Design **three overloaded** versions of a method named `loyaltyPoints` for the cafe:

1. `loyaltyPoints(double amountSpent)` — 1 point per whole dollar spent
2. `loyaltyPoints(double amountSpent, boolean isMember)` — same as above, but members earn double points
3. `loyaltyPoints(int stampCard)` — a completely different rule: a filled stamp card (10 stamps) is worth a flat 50 points, otherwise 0

All three share the name `loyaltyPoints`. Write all three, then call each one from `main` and print the result.

In [ ]:
public class MyLoyaltyPoints {

    // TODO 1: loyaltyPoints(double amountSpent)

    // TODO 2: loyaltyPoints(double amountSpent, boolean isMember)

    // TODO 3: loyaltyPoints(int stampCard)

    public static void main(String[] args) {

        // TODO 4: call all three overloads and print each result

    }
}

MyLoyaltyPoints.main(null);

<details>
<summary><b>One possible solution</b></summary>

```java
public class MyLoyaltyPoints {

    public static int loyaltyPoints(double amountSpent) {
        return (int) amountSpent;
    }

    public static int loyaltyPoints(double amountSpent, boolean isMember) {
        int basePoints = (int) amountSpent;
        return isMember ? basePoints * 2 : basePoints;
    }

    public static int loyaltyPoints(int stampCard) {
        return stampCard >= 10 ? 50 : 0;
    }

    public static void main(String[] args) {
        System.out.println("Spent $17.50:            " + loyaltyPoints(17.50));
        System.out.println("Spent $17.50, member:     " + loyaltyPoints(17.50, true));
        System.out.println("Stamp card with 10:       " + loyaltyPoints(10));
        System.out.println("Stamp card with 6:        " + loyaltyPoints(6));
    }
}

MyLoyaltyPoints.main(null);
```

Three distinct signatures: `loyaltyPoints(double)`, `loyaltyPoints(double, boolean)`, and `loyaltyPoints(int)`. Every call above is unambiguous, because no two signatures share both a parameter count **and** matching types.
</details>

---

# Self-Check: AP-style questions

**1.** Which best describes a method's signature?

&nbsp;&nbsp;(A) Its access modifier and return type
&nbsp;&nbsp;(B) Its name and the types and order of its parameters
&nbsp;&nbsp;(C) Its name and the names of its parameters
&nbsp;&nbsp;(D) Its entire header, including the return type

<details><summary>Answer</summary>

**(B)**. A signature is the name plus the parameter *types*, in order. Parameter names and the return type are both part of the header, but not part of the signature.
</details>

---

**2.** A class already declares `public static int score(int x)`. Which of the following can be added **without** a compile-time error?

&nbsp;&nbsp;(A) `public static double score(int x)`
&nbsp;&nbsp;(B) `public static int score(int y)`
&nbsp;&nbsp;(C) `public static int score(double x)`
&nbsp;&nbsp;(D) `private static int score(int x)`

<details><summary>Answer</summary>

**(C)**. `score(double)` has a different signature than `score(int)`, so it's a valid overload. (A) and (D) change only the return type or the access modifier — the signature `score(int)` is unchanged, so both cause "method already defined." (B) renames a parameter, which doesn't change the signature either.
</details>

---

**3.** What is printed?

```java
public static void announce(String msg) {
    System.out.println(msg);
}

public static void main(String[] args) {
    String result = announce("Order ready!");
}
```

&nbsp;&nbsp;(A) `Order ready!`
&nbsp;&nbsp;(B) `null`
&nbsp;&nbsp;(C) A compile-time error
&nbsp;&nbsp;(D) A run-time error

<details><summary>Answer</summary>

**(C)**. `announce` is declared `void`, so it returns nothing. Trying to store that in `String result` is a type mismatch the compiler catches immediately — the program never runs.
</details>

---

**4.** Given `public static double average(double a, double b)`, which call is valid?

&nbsp;&nbsp;(A) `average(4, 6)`
&nbsp;&nbsp;(B) `average(4.0, 6.0, 8.0)`
&nbsp;&nbsp;(C) `average("4", "6")`
&nbsp;&nbsp;(D) `average(4.0)`

<details><summary>Answer</summary>

**(A)**. The `int` arguments `4` and `6` widen automatically to `double`, so this matches `average(double, double)`. (B) supplies the wrong number of arguments, (C) supplies `String`s which cannot widen to `double`, and (D) supplies too few arguments.
</details>

---

**5.** What is the output?

```java
public static int bonus(int base) { return base + 5; }
public static double bonus(double base) { return base + 5.5; }

public static void main(String[] args) {
    System.out.println(bonus(10));
    System.out.println(bonus(10.0));
}
```

&nbsp;&nbsp;(A) `15` then `15.5` &nbsp;&nbsp; (B) `15.0` then `15.5` &nbsp;&nbsp; (C) `15` then `15.0` &nbsp;&nbsp; (D) A compile-time error, because both methods share the name `bonus`

<details><summary>Answer</summary>

**(A)**. `bonus(10)` — an `int` literal — matches `bonus(int)` exactly, giving `15`. `bonus(10.0)` — a `double` literal — matches `bonus(double)`, giving `15.5`. (D) is wrong because these two methods have different signatures, `bonus(int)` and `bonus(double)`; sharing a name is exactly what overloading allows.
</details>

---

**6.** A method `applyTax(double price)` returns the price with tax added. A programmer writes:

```java
applyTax(19.99);
System.out.println("Total: $19.99");
```

What is the most likely problem?

&nbsp;&nbsp;(A) A compile-time error, since the return value isn't used
&nbsp;&nbsp;(B) A run-time error
&nbsp;&nbsp;(C) A logic error — the computed value with tax was discarded and never printed
&nbsp;&nbsp;(D) There is no problem; this code is correct

<details><summary>Answer</summary>

**(C)**. `applyTax` returns a value, and this code never captures it — the program compiles and runs, but the printed total doesn't include tax. This is a logic error: no error message anywhere, just a wrong result.
</details>

---

**7.** Which two method headers can legally coexist in the same class?

&nbsp;&nbsp;(A) `void report(int x)` and `int report(int x)`
&nbsp;&nbsp;(B) `int report(int x)` and `int report(int y)`
&nbsp;&nbsp;(C) `int report(int x)` and `int report(int x, int y)`
&nbsp;&nbsp;(D) `int report(int x)` and `double report(int x)`

<details><summary>Answer</summary()>

**(C)**. `report(int)` and `report(int, int)` have different parameter counts, so they're genuinely different signatures — valid overloading. (A) and (D) both change only the return type while keeping the signature `report(int)`, which does not create a new overload. (B) only renames a parameter, which the signature ignores entirely.
</details>

---

# Closing time

### Vocabulary to know cold

| Term | One-line definition |
|---|---|
| Method header | The full first line: modifier, return type, name, parameter list |
| Signature | A method's name plus its parameter *types*, in order — nothing else |
| Parameter | A variable declared in a method's header, describing what it needs |
| Argument | The actual value supplied at a call site |
| `void` | A return type meaning the method hands nothing back |
| Overloading | Multiple methods sharing a name but having different signatures |

### The five things that will show up on the exam

1. A **signature** is name + parameter **types**, in order. Not the return type. Not parameter names.
2. Arguments are matched to parameters by **position**, not by name.
3. A `void` method's result can **never** be stored in a variable.
4. A non-`void` return value that's ignored still compiles — and is a **logic error** if it mattered.
5. Overloads must differ in **parameter list**. Differing only in return type is not enough.

### Before you submit, check that you:

- [ ] Ran every code cell, including the four that fail on purpose
- [ ] Labeled every piece of the header and wrote the signature in Hack 1
- [ ] Predicted all four calls in Hack 2 **before** running it
- [ ] Fixed all three bugs in Hack 3 and explained each one
- [ ] Wrote all three `loyaltyPoints` overloads in Hack 4 and confirmed each call is unambiguous
- [ ] Attempted all seven self-check questions before revealing answers

### Next up

**1.10 — Calling Class Methods.** You've called `Math.sqrt(...)` and `Integer.MAX_VALUE` a dozen times without asking why they're written as `ClassName.thing` instead of just `thing`. Next shift covers exactly that: calling a method or reaching a constant through its class name, and what `static` was actually promising you this whole time.

See you at the next shift 🧋